#### This data set consists of positions and absorbed power outputs of wave energy converters (WECs) in four real wave scenarios from the southern coast of Australia.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

## Data Cleaning and Preparations

In [ ]:
df = pd.read_csv('Sydney_Data.csv')
labels = [f"X{i+1}" for i in range(16)] + [f"Y{i+1}" for i in range(16)]  + [f"P{i+1}" for i in range(16)] + ["Total Power"]
df.columns = labels
df.dropna()
before = len(df)
df = df.dropna()
after = len(df)
print(f"{before} rows before removal, {after} rows after removal due to NaN values")

There is no mising values in dataset 

X1-X16 - the posision X of Converter

Y1-Y16 - the position Y of Converter

P1-P16 - Power absorbed 

Total power - output of the farm: Powerall

In [ ]:
df.head()

In [ ]:
df.describe()

### X , Y Histograms 

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(18, 16))
for i, ax in enumerate(axes.flatten()):
    df[f"X{i+1}"].plot(kind='hist', bins=30, alpha=0.5, ax=ax)
    ax.set_title(f'Histogram for X{i+1}')
    ax.set_xlabel('Value')
    ax.set_ylabel('Frequency')
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(4, 4, figsize=(18, 16))
for i, ax in enumerate(axes.flatten()):
    df[f"Y{i+1}"].plot(kind='hist', bins=30, alpha=0.5, ax=ax)
    ax.set_title(f'Histogram for Y{i+1}')
    ax.set_xlabel('Value')
    ax.set_ylabel('Frequency')
plt.tight_layout()
plt.show()


#### Both Feature Y and Feature X exhibit a significant number of outliers concentrated at the extremes of their respective ranges. This suggests that the area within which the converters were operating was constrained. Consequently, due to ocean currents, there were numerous instances where the converters were positioned at the boundaries of the interval.

In [ ]:
x = df['X1']
y = df['Y1']
sns.set_theme(style="ticks")
sns.jointplot(x=x, y=y, kind="hex",color = "#0018E0")
plt.show()

#### The visualization above shows the frequency of measurements recorded by Converter 1 within the given space.

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(18, 16))
for i, ax in enumerate(axes.flatten()):
    df[f"P{i+1}"].plot(kind='hist', bins=30, alpha=0.5, ax=ax)
    ax.set_title(f'Histogram for P{i+1}')
    ax.set_xlabel('Value')
    ax.set_ylabel('Frequency')   
plt.tight_layout()
plt.show()
plt.tight_layout()
plt.show()

In [ ]:
df["Total Power"].hist(bins=50)

#### The histograms of Energy values suggest the need to remove outliers.

In [ ]:
# Calculate Q1 (25th percentile) and Q3 (75th percentile)
Q1 = df['Total Power'].quantile(0.25)
Q3 = df['Total Power'].quantile(0.75)
IQR = Q3 - Q1
# Define the lower and upper bounds for outliers
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
# Filter out the outliers
df_filtered = df[(df['Total Power'] >= lower_bound) & (df['Total Power'] <= upper_bound)]
# Print the number of rows before and after filtering
print(f"Number of rows before filtering: {len(df)}")
print(f"Number of rows after filtering: {len(df_filtered)}")
df = df_filtered

In [ ]:
df["Total Power"].hist(bins=50)

#### The histograms of Energy values after remove outliers.

### Correration

In [ ]:
all_columns = labels
corr_all = df[all_columns].corr()

# Plot the correlation heatmap
plt.figure(figsize=(33, 26))
sns.heatmap(corr_all, annot=True, fmt=".2f", cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlation Matrix for X, Y, and P Columns')
plt.show()

In [ ]:
tab = corr_all.to_numpy().reshape(-1)
plt.hist(tab, bins = 100)
plt.show()

In [ ]:
power_columns = [f"P{i+1}" for i in range(16)] + ["Total Power"]
corr = df[power_columns].corr()
plt.figure(figsize=(30, 24))
sns.heatmap(corr, annot=True, fmt=".2f", cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlation Matrix for P and Total Power Columns')
plt.show()

#### correlation is observed between P1-P16 and Total Power, based on the correlation histogram. Correlations above 0.2 can be considered significant. For this reason, the P1-P16 values are not needed, as we want to avoid training our model on output data. This analysis confirms that P1-P16 are indeed output data.

In [ ]:
new_labels = labels = [f"X{i+1}" for i in range(16)] + [f"Y{i+1}" for i in range(16)] + ["Total Power"]
df = df[new_labels]
df.head()

#### Create Train , Test and Validation set

In [ ]:
X = df.iloc[:,1:-1].values
y = df['Total Power'].values
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=44)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=44)